# 04B — Base analítica (D14)
**TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009–2023**

**Por que existe (D14).** Cada regra de tratamento é calculada **uma vez, aqui**; as rotinas 05 a 09 e 12 apenas leem
esta base. Antes, cada rotina recalculava UF, valores implausíveis, faixa de lote e preço do item do seu jeito.
A rotina 10 continua refazendo tudo por conta própria, como conferência independente.

**O que esta rotina faz (cada passo com a sua decisão no Registro de decisões):**
1. Lê a base pública (rotinas 03 e 03B) de 2009 a 2023 (D2), com ano e mês da data da compra (D22); 2023 entra inteiro (D21).
2. Define o item como código CATMAT + unidade (D17).
3. Deflaciona preço e valor pelo IPCA, base dez/2023 (D20).
4. Classe CATMAT de cada código: a do BPS 2018–2020; para os demais, a do catálogo oficial datado (D40, D41).
   Saem os códigos fora das 16 classes de saúde do Ministério da Saúde (D41).
5. Régua de valores implausíveis sobre o **preço real** (D16, D7): a compra marcada sai de preço e valor, mas fica nas contagens.
6. Colunas de referência do item no ano: mediana do preço real, número de compras, faixa de lote em quintis (D31),
   esta só em item-ano com 3 compras válidas ou mais (D53).

**Entradas:** `Base Harmonizacao Publica`, `Base Harmonizacao Campos` (classe 2018–2020), `Base Classificacao CATMAT/classe_catalogo.parquet`
(rotina 03D), `IPCA Tratado`. **Saídas:** `Base Analitica/base_analitica.parquet` e registros `Auditoria/BA1` e `BA2`.

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # fora do Colab: use a variável de ambiente BPS_DADOS
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')
# ─────────────────────────────────────────────────────────────────────────
!pip install polars==1.35.2 pyarrow --quiet
import polars as pl
from pathlib import Path
pl.Config.set_tbl_rows(30); pl.Config.set_tbl_width_chars(200)

BASE            = Path(PASTA_DADOS)
P_PUB           = BASE / 'Base Harmonizacao Publica'
P_HARM          = BASE / 'Base Harmonizacao Campos'
ARQ_CATALOGO    = BASE / 'Base Classificacao CATMAT' / 'classe_catalogo.parquet'   # rotina 03D, consulta de out/2026 (D41)
ARQ_IPCA        = BASE / 'IPCA Tratado' / 'ipca_deflator.parquet'
SAIDA           = BASE / 'Base Analitica'
PASTA_AUDITORIA = BASE / 'Auditoria'
for p in (SAIDA, PASTA_AUDITORIA):
    p.mkdir(parents=True, exist_ok=True)
for p in (P_PUB, P_HARM, ARQ_CATALOGO, ARQ_IPCA):
    assert p.exists(), f'Não encontrado: {p}'

ANO_INICIO, ANO_FIM = 2009, 2023                  # D2: índice de preço 2009–2023
ANOS_COM_CLASSE_NO_BPS = (2018, 2019, 2020)       # D40: únicos arquivos do BPS com a coluna de classe
# D41: classes de saúde publicadas pelo Ministério da Saúde ("Passo a passo do BPS: consulta de itens")
CLASSES_SAUDE_MS = ['6505', '6508', '6509', '6510', '6515', '6520', '6525', '6530', '6532',
                    '6540', '6545', '6550', '6640', '6810', '6840', '8940']
# D7/D16: régua de valores implausíveis — preço 20× acima ou abaixo, ou quantidade 200× acima da mediana do item no ano,
# só em item-ano com 3 compras ou mais (IN SEGES/ME 65/2021, art. 6º: "três ou mais preços", como analogia; teste 04b)
REGUA_PRECO, REGUA_QTD, MIN_REG_REGUA = 20, 200, 3
QUINTIS = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']          # D31: quintis como convenção única de grupos
MIN_MEDIANA = 3                                   # D45/D53: mediana do item no ano só com 3 preços ou mais (IN SEGES/ME 65/2021, art. 6º, analogia)
print('Pronto ✓')

Mounted at /content/drive
Pronto ✓


## 1. Leitura, ano e mês da compra, item e deflação (D2, D22, D17, D20)

In [2]:
COLS = ['data_compra', 'cod_catmat', 'desc_item', 'unidade_chave', 'unidade_rotulo', 'unidade_marca_generico',
        'preco_unitario', 'quantidade', 'valor_total', 'modalidade_compra', 'modalidade_grupo',
        'uf', 'municipio', 'local_recuperado', 'origem_local', 'cnpj_instituicao', 'instituicao',
        'cnpj_fornecedor', 'fornecedor', 'natureza_cod', 'natureza_desc', 'copias_retiradas']
fr = []
for a in sorted(P_PUB.glob('BPS_*.parquet')):
    disp = pl.read_parquet(a, n_rows=0).columns
    fr.append(pl.read_parquet(a, columns=[c for c in COLS if c in disp]))
pub = pl.concat(fr, how='diagonal')
ipca = pl.read_parquet(ARQ_IPCA).select('ano', 'mes', 'fator_deflacao')

base = (pub.filter(pl.col('data_compra').is_not_null() & pl.col('data_compra').dt.year().is_between(ANO_INICIO, ANO_FIM)
                   & (pl.col('preco_unitario') > 0) & pl.col('cod_catmat').is_not_null())
           .with_columns(pl.col('data_compra').dt.year().cast(pl.Int32).alias('ano'),          # D22: ano e mês da data da compra
                         pl.col('data_compra').dt.month().cast(pl.Int8).alias('mes'),
                         (pl.col('cod_catmat') + '|' + pl.col('unidade_chave').fill_null('')).alias('item'))   # D17
           .join(ipca, on=['ano', 'mes'], how='left')
           .with_columns((pl.col('preco_unitario') * pl.col('fator_deflacao')).alias('preco_real'),   # D20: reais de dez/2023
                         (pl.col('valor_total') * pl.col('fator_deflacao')).alias('valor_real')))
assert base['fator_deflacao'].null_count() == 0, 'mês sem deflator'
print(f'Base pública lida: {pub.height:,} | na análise ({ANO_INICIO}–{ANO_FIM}, preço > 0, com código): {base.height:,}')

Base pública lida: 836,096 | na análise (2009–2023, preço > 0, com código): 836,095


## 2. Classe CATMAT e exclusão dos códigos fora da saúde (D40, D41)

In [3]:
classe_bps = (pl.concat([pl.read_parquet(P_HARM / f'BPS_{a}.parquet', columns=['cod_catmat', 'classe_catmat'])
                         for a in ANOS_COM_CLASSE_NO_BPS])
                .with_columns(pl.col('classe_catmat').cast(pl.Utf8).str.strip_chars())
                .filter(pl.col('cod_catmat').is_not_null() & pl.col('classe_catmat').is_not_null() & (pl.col('classe_catmat') != ''))
                .select('cod_catmat', pl.col('classe_catmat').alias('classe_bps')).unique())
assert classe_bps.group_by('cod_catmat').len().filter(pl.col('len') > 1).height == 0, 'código com mais de uma classe no BPS'
cat = (pl.read_parquet(ARQ_CATALOGO).filter(pl.col('encontrado').fill_null(False))
         .select('cod_catmat', 'classe_catalogo', 'nome_classe'))
nomes = cat.select(pl.col('classe_catalogo').alias('classe'), pl.col('nome_classe')).unique('classe')   # nome oficial de cada classe
base = (base.join(classe_bps, on='cod_catmat', how='left').join(cat.select('cod_catmat', 'classe_catalogo'), on='cod_catmat', how='left')
            .with_columns(pl.coalesce('classe_bps', 'classe_catalogo').alias('classe_catmat'),
                          pl.when(pl.col('classe_bps').is_not_null()).then(pl.lit('BPS 2018-2020'))
                            .when(pl.col('classe_catalogo').is_not_null()).then(pl.lit('catalogo, consulta out/2026'))
                            .otherwise(None).alias('origem_classe'))
            .drop('classe_bps', 'classe_catalogo')
            .join(nomes.rename({'classe': 'classe_catmat'}), on='classe_catmat', how='left'))
sem_classe = base.filter(pl.col('classe_catmat').is_null()).height
assert sem_classe == 0, f'{sem_classe:,} compras sem classe: rode a rotina 03D até não haver falhas de consulta'

fora = base.filter(~pl.col('classe_catmat').is_in(CLASSES_SAUDE_MS))
ba1 = (base.group_by('ano').agg(pl.len().alias('compras_lidas'),
                                (pl.col('origem_classe') == 'BPS 2018-2020').sum().alias('classe_do_bps'),
                                (pl.col('origem_classe') == 'catalogo, consulta out/2026').sum().alias('classe_do_catalogo'),
                                (~pl.col('classe_catmat').is_in(CLASSES_SAUDE_MS)).sum().alias('fora_classes_saude_retiradas'))
           .sort('ano'))
base = base.filter(pl.col('classe_catmat').is_in(CLASSES_SAUDE_MS))
ba1 = ba1.with_columns((pl.col('compras_lidas') - pl.col('fora_classes_saude_retiradas')).alias('compras_na_base_analitica'))
print(ba1); print(f'Retiradas (fora das classes de saúde, D41): {fora.height:,} compras de {fora["cod_catmat"].n_unique()} códigos')
ba1.write_csv(PASTA_AUDITORIA / 'BA1_classe_e_exclusao_por_ano.csv')
print('Registro: Auditoria/BA1_classe_e_exclusao_por_ano.csv')

shape: (15, 6)
┌──────┬───────────────┬───────────────┬────────────────────┬──────────────────────────────┬───────────────────────────┐
│ ano  ┆ compras_lidas ┆ classe_do_bps ┆ classe_do_catalogo ┆ fora_classes_saude_retiradas ┆ compras_na_base_analitica │
│ ---  ┆ ---           ┆ ---           ┆ ---                ┆ ---                          ┆ ---                       │
│ i32  ┆ u32           ┆ u32           ┆ u32                ┆ u32                          ┆ u32                       │
╞══════╪═══════════════╪═══════════════╪════════════════════╪══════════════════════════════╪═══════════════════════════╡
│ 2009 ┆ 5111          ┆ 4453          ┆ 658                ┆ 6                            ┆ 5105                      │
│ 2010 ┆ 10072         ┆ 9118          ┆ 954                ┆ 9                            ┆ 10063                     │
│ 2011 ┆ 8306          ┆ 7489          ┆ 817                ┆ 7                            ┆ 8299                      │
│ 2012 ┆ 3217    

## 3. Régua de valores implausíveis sobre o preço real (D16, D7)

In [4]:
cnpj = pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars()
avaliavel = (cnpj.is_not_null() & (cnpj.str.len_chars() > 0) & (cnpj != '00000000000000')
             & (pl.col('quantidade') > 0) & (pl.col('valor_total') > 0))
base = base.with_row_index('id_compra')
marca = (base.filter(avaliavel)
            .with_columns(pl.col('preco_real').median().over(['item', 'ano']).alias('_p_med'),
                          pl.col('quantidade').median().over(['item', 'ano']).alias('_q_med'),
                          pl.len().over(['item', 'ano']).alias('_n'))
            .with_columns((((pl.col('preco_real') / pl.col('_p_med') > REGUA_PRECO) | (pl.col('preco_real') / pl.col('_p_med') < 1 / REGUA_PRECO)
                            | (pl.col('quantidade') / pl.col('_q_med') > REGUA_QTD)) & (pl.col('_n') >= MIN_REG_REGUA)).alias('valor_implausivel'))
            .select('id_compra', 'valor_implausivel'))
base = base.join(marca, on='id_compra', how='left')   # nulo = sem fornecedor, quantidade ou valor: não avaliável pela régua
ba2 = (base.group_by('ano').agg(pl.len().alias('compras'),
                                (pl.col('valor_implausivel') == True).sum().alias('implausiveis'),
                                pl.col('valor_implausivel').is_null().sum().alias('nao_avaliaveis'))
           .with_columns((pl.col('implausiveis') / pl.col('compras') * 100).round(2).alias('pct_implausiveis')).sort('ano'))
print(ba2); print(f'Implausíveis: {int(ba2["implausiveis"].sum()):,} | não avaliáveis: {int(ba2["nao_avaliaveis"].sum()):,}')
ba2.write_csv(PASTA_AUDITORIA / 'BA2_regua_por_ano.csv')
print('Registro: Auditoria/BA2_regua_por_ano.csv')

shape: (15, 5)
┌──────┬─────────┬──────────────┬────────────────┬──────────────────┐
│ ano  ┆ compras ┆ implausiveis ┆ nao_avaliaveis ┆ pct_implausiveis │
│ ---  ┆ ---     ┆ ---          ┆ ---            ┆ ---              │
│ i32  ┆ u32     ┆ u32          ┆ u32            ┆ f64              │
╞══════╪═════════╪══════════════╪════════════════╪══════════════════╡
│ 2009 ┆ 5105    ┆ 77           ┆ 0              ┆ 1.51             │
│ 2010 ┆ 10063   ┆ 300          ┆ 0              ┆ 2.98             │
│ 2011 ┆ 8299    ┆ 331          ┆ 0              ┆ 3.99             │
│ 2012 ┆ 3214    ┆ 39           ┆ 0              ┆ 1.21             │
│ 2013 ┆ 2169    ┆ 19           ┆ 0              ┆ 0.88             │
│ 2014 ┆ 19384   ┆ 193          ┆ 0              ┆ 1.0              │
│ 2015 ┆ 97587   ┆ 2513         ┆ 0              ┆ 2.58             │
│ 2016 ┆ 89188   ┆ 4097         ┆ 0              ┆ 4.59             │
│ 2017 ┆ 86018   ┆ 3318         ┆ 0              ┆ 3.86             │
│ 201

## 4. Referência do item no ano e faixa de lote (D16, D31)
> Duas contagens, de propósito: `compras_item_ano` conta **todas** as compras do item no ano (contagens e HHI por compras,
> D16); `compras_validas_item_ano` conta as que entram em preço e valor (sem implausíveis), base dos mínimos das medianas
> e percentis (D44, D45). A mediana e a faixa de lote usam só as compras válidas. A faixa de lote só é atribuída em item-ano
> com 3 compras válidas ou mais (D53, o mesmo mínimo da D45); nos demais fica vazia.

In [5]:
# inclui as não avaliáveis pela régua (nulo = sem fornecedor, quantidade ou valor). fill_null(False) é necessário:
# no Polars, 'nulo != True' dá nulo e a linha sairia do filtro
valida = ~pl.col('valor_implausivel').fill_null(False)
ref = (base.filter(valida & (pl.col('quantidade') > 0))
           .with_columns(pl.col('preco_real').median().over(['item', 'ano']).alias('mediana_item_ano'),
                         pl.len().over(['item', 'ano']).alias('compras_validas_item_ano'),
                         (pl.col('quantidade').log() - pl.col('quantidade').log().median().over(['item', 'ano'])).alias('_dq'))
           # D53: a faixa de lote só existe em item-ano com 3 compras válidas ou mais; abaixo disso a compra seria comparada
           # consigo mesma (desvio zero) ou com preços que a régua não confere. Os quintis são calculados só entre essas compras.
           .with_columns(pl.when(pl.col('compras_validas_item_ano') >= MIN_MEDIANA).then(pl.col('_dq')).otherwise(None)
                           .qcut(5, labels=QUINTIS, allow_duplicates=True).cast(pl.Utf8).alias('faixa_lote'),
                         pl.col('_dq').exp().alias('lote_rel'))
           .select('id_compra', 'mediana_item_ano', 'compras_validas_item_ano', 'faixa_lote', 'lote_rel'))
base = (base.join(ref, on='id_compra', how='left')
            .with_columns(pl.len().over(['item', 'ano']).alias('compras_item_ano'),
                          (pl.col('preco_real') / pl.col('mediana_item_ano')).alias('preco_rel_mediana')))
print(f'Compras com referência do item no ano: {base["mediana_item_ano"].is_not_null().sum():,} de {base.height:,}')
print(f'Com faixa de lote (item-ano com {MIN_MEDIANA} compras válidas ou mais, D53): {base["faixa_lote"].is_not_null().sum():,}')

Compras com referência do item no ano: 802,660 de 833,928
Com faixa de lote (item-ano com 3 compras válidas ou mais, D53): 747,684


## 5. Gravação e conferência

In [6]:
COLS_SAIDA = ['id_compra', 'data_compra', 'ano', 'mes', 'item', 'cod_catmat', 'unidade_chave', 'unidade_rotulo', 'unidade_marca_generico',
              'desc_item', 'classe_catmat', 'nome_classe', 'origem_classe', 'cnpj_instituicao', 'instituicao', 'natureza_cod', 'natureza_desc',
              'uf', 'municipio', 'local_recuperado', 'origem_local', 'cnpj_fornecedor', 'fornecedor', 'modalidade_grupo', 'modalidade_compra',
              'preco_unitario', 'quantidade', 'valor_total', 'fator_deflacao', 'preco_real', 'valor_real', 'valor_implausivel',
              'compras_item_ano', 'compras_validas_item_ano', 'mediana_item_ano', 'preco_rel_mediana', 'faixa_lote', 'lote_rel',
              'copias_retiradas']
saida = base.select([c for c in COLS_SAIDA if c in base.columns]).sort('id_compra')
assert saida['id_compra'].is_unique().all()
assert saida.height == int(ba1['compras_na_base_analitica'].sum()), 'contagem não confere com o registro BA1'
saida.write_parquet(SAIDA / 'base_analitica.parquet')
print(f'Base analítica: {saida.height:,} compras | {saida["item"].n_unique():,} itens | {saida["cnpj_instituicao"].n_unique():,} compradores')
print(f'Gravado: Base Analitica/base_analitica.parquet ({len(saida.columns)} colunas)')

Base analítica: 833,928 compras | 19,979 itens | 2,027 compradores
Gravado: Base Analitica/base_analitica.parquet (39 colunas)
